# Train SymGate-VDSR

Wires together:
- `model_symgate_vdsr.py`
- `dataloader_anchor.py` (unchanged)
- `hann_merge.py` (unchanged)

### Key differences vs `train_lidar.ipynb` (DG-VDSR)

| | DG-VDSR | SymGate-VDSR |
|---|---|---|
| Optimisers | Two (stream_a / stream_b) | One (all params) |
| Scheduler | ReduceLROnPlateau per stream | CosineAnnealingWarmRestarts |
| LR floor | 1e-6 (hit epoch 140/272) | 1e-5 + warm restarts |
| Lambda_pin ramp | 1->5 over 15 epochs | Not needed (structural fix) |
| Anchor metric | val_anchor_mae on dem_pred | val_aux_mae (Stream B head) |
| val_anchor_mae | meaningful | ~0 by CSPN construction (sanity only) |
| Grad norm split | gA / gB | dense / sparse / fusion / refine / head |
| Loss forward args | (pred, gt, lidar_raw, mask, dist) | (outputs_dict, gt, lidar_raw, lidar_delta, mask, dist) |
| model() return | (dem_pred, alpha, r_topo, r_anchor) | dict with dem_pred, dem_coarse, r_anchor_aux, ... |

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, math, time, gc, shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from dataloader_anchor import create_dataloaders
from model_symgatevdsr import (
    SymGateVDSR, SymGateTopoLoss,
    build_recommended_scheduler, branch_grad_norms,
)
from hann_merge import HannStreamMerger

# Detect environment
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_COLAB
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.backends.cudnn.benchmark = True

# =============================================================================
# LOCAL WORKSTATION (P4000)
# =============================================================================
if IN_LOCAL:
    PROJECT_DIR = Path(r'D:\Vaibhav\vdsr-atl08')   # <-- edit if different
    DATASET_DIR = PROJECT_DIR / 'Dataset'
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_DIR}.\n'
            f'Download it and place it at {DATASET_DIR}, or update PROJECT_DIR above.'
        )
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    print(f'Dataset found: {DATASET_DIR}')
    print(f'Scripts:       {PROJECT_DIR}')

# =============================================================================
# GOOGLE COLAB (T4)
# =============================================================================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATASET = Path('/content/drive/MyDrive/dem-lidar/Dataset')
    LOCAL_DATASET = Path('/content/dataset')
    if not LOCAL_DATASET.exists():
        if DRIVE_DATASET.exists():
            print('Copying dataset from Drive to local SSD...')
            shutil.copytree(str(DRIVE_DATASET), str(LOCAL_DATASET))
            print('Done.')
        else:
            raise FileNotFoundError(
                f'Dataset not found at {DRIVE_DATASET}.\n'
                f'Upload your dataset to Google Drive at that path first.'
            )
    else:
        print(f'Local dataset already exists: {LOCAL_DATASET}')
    SCRIPT_DIR = Path('/content')
    if str(SCRIPT_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPT_DIR))


Environment: Local
Using device: cuda
Dataset found: D:\Vaibhav\vdsr-atl08\Dataset
Scripts:       D:\Vaibhav\vdsr-atl08


In [2]:
from pathlib import Path

# ---- Paths ------------------------------------------------------------------
if IN_LOCAL:
    BASE_DIR       = str(DATASET_DIR)
    CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_symgate_vdsr'

if IN_COLAB:
    BASE_DIR       = str(LOCAL_DATASET)
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/checkpoints_symgate_vdsr')

TRAIN_DIRS = [
    f"{BASE_DIR}/Kl/tensors/train",
    f"{BASE_DIR}/SG/tensors/train",
]
VAL_DIRS = [
    f"{BASE_DIR}/Kl/tensors/validation_contiguous",
    f"{BASE_DIR}/SG/tensors/validation_contiguous",
]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "symgate_vdsr"

# ---- Model ------------------------------------------------------------------
FEATURES   = 64
NUM_GROUPS = 8
CSPN_ITERS = 6

# ---- Optimiser & LR ---------------------------------------------------------
# Single optimiser -- no stream_a / stream_b split.
# The symmetric gate design ensures equal gradient magnitude at init without
# per-branch LR tuning.
LR             = 1e-4
WEIGHT_DECAY   = 5e-5
GRAD_CLIP_NORM = 1.0

# Linear warmup: ~12 epochs before cosine takes over.
WARMUP_STEPS = 12 * (1649 // 16) + 16   # approx 12 epochs x ~103 batches

# ---- Scheduler (CosineAnnealingWarmRestarts) ---------------------------------
# Replaces ReduceLROnPlateau so the run does not get frozen by a gridlocked
# loss surface that looks like convergence.
COSINE_T0      = 40     # restart period in epochs
COSINE_T_MULT  = 1      # keep period constant (2 = double each cycle)
COSINE_ETA_MIN = 1e-5   # floor is 10x higher than old 1e-6

# ---- Loss weights -----------------------------------------------------------
LOSS_ALPHA      = 0.5   # base elevation (optical, halo-masked)
LOSS_BETA       = 2.5   # slope (supervised far / unsupervised halo)
LOSS_GAMMA      = 0.5   # curvature
LOSS_LAMBDA_PIN = 1.0   # pin loss on dem_coarse; no ramp needed
LOSS_LAMBDA_AUX = 0.4   # Stream-B auxiliary delta head
PIN_BETA        = 1.0   # SmoothL1 knee in metres
DECAY_RADIUS    = 15.0  # halo decay width in metres
BUFFER_SIZE     = 3     # buffer ring in pixels (3px x 10m = 30m)
HALO_WEIGHT     = 0.25  # weight of unsupervised halo smoothness

# ---- Data -------------------------------------------------------------------
BATCH_SIZE      = 16
TRAIN_CROP      = 128
VAL_CROP        = 256
VAL_OVERLAP     = 192
NUM_WORKERS     = 8
VAL_PATCH_BATCH = 32    # patches per GPU mini-batch during validation

# ---- Training control -------------------------------------------------------
EPOCHS                  = 500
EARLY_STOPPING_PATIENCE = 500
# Composite = val_slope_rmse + val_aux_mae
# (val_anchor_mae on dem_pred is always ~0 by CSPN construction -- not useful)
BEST_METRIC_NAME = "composite"

training_config = {
    'run_name': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'warmup_steps': WARMUP_STEPS,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'features': FEATURES,
    'num_groups': NUM_GROUPS,
    'cspn_iters': CSPN_ITERS,
    'loss_alpha': LOSS_ALPHA,
    'loss_beta': LOSS_BETA,
    'loss_gamma': LOSS_GAMMA,
    'loss_lambda_pin': LOSS_LAMBDA_PIN,
    'loss_lambda_aux': LOSS_LAMBDA_AUX,
    'pin_beta': PIN_BETA,
    'decay_radius': DECAY_RADIUS,
    'buffer_size': BUFFER_SIZE,
    'halo_weight': HALO_WEIGHT,
    'cosine_t0': COSINE_T0,
    'cosine_t_mult': COSINE_T_MULT,
    'cosine_eta_min': COSINE_ETA_MIN,
    'train_crop': TRAIN_CROP,
    'val_crop': VAL_CROP,
    'val_overlap': VAL_OVERLAP,
    'val_patch_batch': VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name': BEST_METRIC_NAME,
    'train_dirs': TRAIN_DIRS,
    'val_dirs': VAL_DIRS,
}
print(json.dumps(training_config, indent=2))


{
  "run_name": "symgate_vdsr",
  "epochs": 500,
  "batch_size": 16,
  "lr": 0.0001,
  "warmup_steps": 1252,
  "weight_decay": 5e-05,
  "grad_clip_norm": 1.0,
  "features": 64,
  "num_groups": 8,
  "cspn_iters": 6,
  "loss_alpha": 0.5,
  "loss_beta": 2.5,
  "loss_gamma": 0.5,
  "loss_lambda_pin": 1.0,
  "loss_lambda_aux": 0.4,
  "pin_beta": 1.0,
  "decay_radius": 15.0,
  "buffer_size": 3,
  "halo_weight": 0.25,
  "cosine_t0": 40,
  "cosine_t_mult": 1,
  "cosine_eta_min": 1e-05,
  "train_crop": 128,
  "val_crop": 256,
  "val_overlap": 192,
  "val_patch_batch": 32,
  "early_stopping_patience": 500,
  "best_metric_name": "composite",
  "train_dirs": [
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/Kl/tensors/train",
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/SG/tensors/train"
  ],
  "val_dirs": [
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/Kl/tensors/validation_contiguous",
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/SG/tensors/validation_contiguous"
  ]
}


In [3]:
# ---- Metric helpers ---------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32
).view(1, 1, 3, 3)

def get_warmup_lr(step, base_lr, warmup_steps):
    if step >= warmup_steps:
        return base_lr
    return base_lr * (step + 1) / warmup_steps

# ---- Checkpoint -------------------------------------------------------------
def save_checkpoint(
    epoch, model, optimizer, scheduler,
    best_metric, best_epoch, train_history, checkpoint_dir, training_config, is_best=False
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'composite'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
    }
    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, filename)
    meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as fh:
        json.dump({
            'epoch': epoch,
            'best_metric': best_metric,
            'best_epoch': best_epoch,
            'training_config': training_config,
            'torch_version': torch.__version__,
        }, fh, indent=2)
    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        torch.save(ckpt, best_path)
    return filename

# ---- train_one_epoch --------------------------------------------------------
# Single-optimiser loop for SymGateVDSR.
#
# Differences vs old DG-VDSR loop:
#   model() returns a dict; criterion() needs lidar_delta as extra arg.
#   branch_grad_norms() replaces the old gA/gB split (now 5 keys).
#   scheduler.step() called per batch (CAWR), not per epoch.
# -----------------------------------------------------------------------------
def train_one_epoch(
    model, criterion, optimizer, scheduler, train_loader, device,
    grad_clip_norm=1.0, epoch=0, warmup_steps=0, base_lr=1e-4,
    log_grad_norms=True,
):
    model.train()

    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0, 'curve': 0.0,
        'pin': 0.0, 'aux': 0.0,
        'mae': 0.0, 'rmse': 0.0,
        'gn_dense': 0.0, 'gn_sparse': 0.0, 'gn_fusion': 0.0,
        'gn_refine': 0.0, 'gn_head': 0.0,
    }
    n_batches = 0
    global_step_offset = (epoch - 1) * len(train_loader)

    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)

    for batch_idx, batch in enumerate(pbar):
        dem_bic     = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask        = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map    = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem      = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_raw   = dem_bic + lidar_delta

        # Linear LR warmup (step-based)
        global_step = global_step_offset + batch_idx
        if global_step < warmup_steps:
            warm_lr = get_warmup_lr(global_step, base_lr, warmup_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = warm_lr

        optimizer.zero_grad(set_to_none=True)

        # Forward -- returns dict: dem_pred, dem_coarse, r_anchor_aux, gate_sparse, affinity
        outputs   = model(dem_bic, lidar_delta, mask, dist_map)
        # Loss -- criterion needs lidar_delta (5th arg) for aux_loss
        loss_dict = criterion(outputs, gt_dem, lidar_raw, lidar_delta, mask, dist_map)

        if not torch.isfinite(loss_dict['total']):
            tqdm.write(f"WARNING: non-finite loss at epoch {epoch}, batch {batch_idx} -- skipping")
            optimizer.zero_grad(set_to_none=True)
            continue

        loss_dict['total'].backward()

        # Branch grad norms -- 5 keys; catch sparse starvation in first few epochs
        if log_grad_norms:
            gn = branch_grad_norms(model)
            running['gn_dense']  += gn['dense_encoder']
            running['gn_sparse'] += gn['sparse_encoder']
            running['gn_fusion'] += gn['fusion']
            running['gn_refine'] += gn['refine']
            running['gn_head']   += gn['head']

        if grad_clip_norm is not None and grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        optimizer.step()

        # CosineAnnealingWarmRestarts steps per batch after warmup
        if global_step >= warmup_steps:
            scheduler.step(epoch - 1 + batch_idx / len(train_loader))

        with torch.no_grad():
            pred_dem   = outputs['dem_pred']
            batch_mae  = F.l1_loss(pred_dem, gt_dem)
            batch_rmse = compute_rmse(pred_dem, gt_dem)

        running['total'] += float(loss_dict['total'].item())
        running['base']  += float(loss_dict['base'].item())
        running['slope'] += float(loss_dict['slope'].item())
        running['curve'] += float(loss_dict['curve'].item())
        running['pin']   += float(loss_dict['pin'].item())
        running['aux']   += float(loss_dict['aux'].item())
        running['mae']   += float(batch_mae.item())
        running['rmse']  += float(batch_rmse.item())
        n_batches += 1

        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'pin':  f"{loss_dict['pin'].item():.4f}",
            'aux':  f"{loss_dict['aux'].item():.4f}",
            'mae':  f"{batch_mae.item():.4f}",
            'lr':   f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    for k in running:
        running[k] /= max(n_batches, 1)
    return running


In [4]:
# ---- validate_one_epoch -----------------------------------------------------
# Validation loop for SymGateVDSR.
#
# Key changes vs old DG-VDSR loop:
#   model() returns a dict; unpack dem_pred and r_anchor_aux.
#   val_anchor_mae on dem_pred is ~0 by CSPN construction -- still logged
#   as a sanity check, but the meaningful sparse-stream diagnostic is
#   val_aux_mae (r_anchor_aux MAE at mask pixels).
#   criterion() needs lidar_delta as 4th positional arg.
# -----------------------------------------------------------------------------
@torch.inference_mode()
def validate_one_epoch(model, criterion, val_loader, device, val_patch_batch=32):
    model.eval()

    sobel_x_cpu   = sobel_x.cpu()
    sobel_y_cpu   = sobel_y.cpu()
    laplacian_cpu = laplacian.cpu()

    total_loss       = 0.0
    total_mae        = 0.0
    total_rmse       = 0.0
    total_psnr       = 0.0
    total_slope_rmse = 0.0
    total_curve_rmse = 0.0
    total_anchor_mae = 0.0   # sanity: expect ~0 throughout
    total_aux_mae    = 0.0   # real Stream B metric
    n_samples        = 0

    outer = tqdm(val_loader, desc='Validate files', leave=False, dynamic_ncols=True)

    for sample_idx, batch in enumerate(outer, start=1):
        # Keep all patches on CPU; move in mini-batches to avoid VRAM explosion
        dem_bic_all     = batch['dem_bic'].squeeze(0).float()
        lidar_delta_all = batch['lidar_delta'].squeeze(0).float()
        mask_all        = batch['mask'].squeeze(0).float()
        dist_map_all    = batch['dist_map'].squeeze(0).float()
        gt_dem_all      = batch['gt_dem'].squeeze(0).float()

        patch_mean_all = batch['patch_mean'].squeeze(0)
        coords_all     = batch['coords'].squeeze(0)
        canvas_shape   = batch['canvas_shape'].squeeze(0).tolist()
        pad            = batch['pad'].item()
        original_shape = batch['original_shape'].squeeze(0).tolist()

        gt_canvas = (
            batch['gt_canvas_full'].squeeze(0).float()
            .unsqueeze(0).unsqueeze(0)
        )

        merger = HannStreamMerger(
            canvas_shape=canvas_shape,
            patch_size=256,
            device='cpu',
            pad=pad,
            original_shape=original_shape,
        )

        patch_count     = dem_bic_all.shape[0]
        n_patch_batches = math.ceil(patch_count / val_patch_batch)

        running_val_loss    = {'total': 0.0}
        file_anchor_mae_sum = 0.0
        file_aux_mae_sum    = 0.0
        file_mask_sum       = mask_all.sum()
        n_val_batches       = 0

        inner = tqdm(
            range(n_patch_batches),
            desc=f'Patches {sample_idx}/{len(val_loader)}',
            leave=False, dynamic_ncols=True,
        )

        for batch_idx in inner:
            start = batch_idx * val_patch_batch
            end   = min(start + val_patch_batch, patch_count)

            dem_bic      = dem_bic_all[start:end].to(device, non_blocking=True)
            lidar_delta  = lidar_delta_all[start:end].to(device, non_blocking=True)
            mask         = mask_all[start:end].to(device, non_blocking=True)
            dist_map     = dist_map_all[start:end].to(device, non_blocking=True)
            gt_dem_batch = gt_dem_all[start:end].to(device, non_blocking=True)

            lidar_raw_batch = dem_bic + lidar_delta

            outputs  = model(dem_bic, lidar_delta, mask, dist_map)
            pred_dem = outputs['dem_pred']       # hard-anchored at ATL08 pixels
            r_aux    = outputs['r_anchor_aux']   # Stream B's own delta prediction

            loss_dict = criterion(
                outputs, gt_dem_batch, lidar_raw_batch, lidar_delta, mask, dist_map
            )
            running_val_loss['total'] += float(loss_dict['total'].item())

            if mask.sum() > 0:
                # anchor MAE on dem_pred -- expect ~0 by CSPN hard-reset
                file_anchor_mae_sum += float(
                    (mask * torch.abs(pred_dem - lidar_raw_batch)).sum().item()
                )
                # aux MAE -- how well does Stream B predict lidar_delta?
                file_aux_mae_sum += float(
                    (mask * torch.abs(r_aux - lidar_delta)).sum().item()
                )

            n_val_batches += 1
            merger.add_batch(
                pred_dem.detach().cpu(),
                patch_mean_all[start:end],
                coords_all[start:end].cpu(),
            )
            inner.set_postfix({'batch': f'{batch_idx + 1}/{n_patch_batches}'})

        val_loss = running_val_loss['total'] / max(n_val_batches, 1)

        if file_mask_sum > 0:
            val_anchor_mae = file_anchor_mae_sum / float(file_mask_sum.item())
            val_aux_mae    = file_aux_mae_sum    / float(file_mask_sum.item())
        else:
            val_anchor_mae = 0.0
            val_aux_mae    = 0.0

        # Stitch and compute full-DEM metrics on CPU
        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)

        del dem_bic_all, lidar_delta_all, mask_all, dist_map_all, gt_dem_all, merger

        mae       = F.l1_loss(merged_pred, gt_canvas)
        rmse      = compute_rmse(merged_pred, gt_canvas)
        psnr      = compute_psnr(merged_pred, gt_canvas)

        pred_dx   = safe_conv(merged_pred, sobel_x_cpu)
        pred_dy   = safe_conv(merged_pred, sobel_y_cpu)
        gt_dx     = safe_conv(gt_canvas,   sobel_x_cpu)
        gt_dy     = safe_conv(gt_canvas,   sobel_y_cpu)
        pred_slope = torch.sqrt(pred_dx ** 2 + pred_dy ** 2)
        gt_slope   = torch.sqrt(gt_dx   ** 2 + gt_dy   ** 2)
        slope_rmse = compute_rmse(pred_slope, gt_slope)

        pred_curve = safe_conv(merged_pred, laplacian_cpu)
        gt_curve   = safe_conv(gt_canvas,   laplacian_cpu)
        curve_rmse = compute_rmse(pred_curve, gt_curve)

        del merged_pred, gt_canvas

        total_loss       += val_loss
        total_mae        += float(mae.item())
        total_rmse       += float(rmse.item())
        total_psnr       += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        total_anchor_mae += val_anchor_mae
        total_aux_mae    += val_aux_mae
        n_samples        += 1

        outer.set_postfix({
            'mae':   f'{mae.item():.3f}',
            'rmse':  f'{rmse.item():.3f}',
            'slope': f'{slope_rmse.item():.3f}',
            'aux':   f'{val_aux_mae:.4f}',
        })

    return {
        'val_total':      total_loss       / max(n_samples, 1),
        'val_mae':        total_mae        / max(n_samples, 1),
        'val_rmse':       total_rmse       / max(n_samples, 1),
        'val_psnr':       total_psnr       / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse / max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse / max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae / max(n_samples, 1),  # sanity ~0
        'val_aux_mae':    total_aux_mae    / max(n_samples, 1),  # real Stream B metric
    }


In [7]:
def main():
    train_loader, val_loader = create_dataloaders(
        TRAIN_DIRS, VAL_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        pin_memory=True,
    )

    model     = SymGateVDSR(features=FEATURES, num_groups=NUM_GROUPS, cspn_iters=CSPN_ITERS).to(device)
    criterion = SymGateTopoLoss(
        alpha        = LOSS_ALPHA,
        beta         = LOSS_BETA,
        gamma        = LOSS_GAMMA,
        lambda_pin   = LOSS_LAMBDA_PIN,
        lambda_aux   = LOSS_LAMBDA_AUX,
        pin_beta     = PIN_BETA,
        decay_radius = DECAY_RADIUS,
        buffer_size  = BUFFER_SIZE,
        halo_weight  = HALO_WEIGHT,
    ).to(device)

    # Single optimiser -- no stream_a / stream_b split
    optimizer = torch.optim.Adam([
    {'params': list(model.dense_encoder.parameters()) +
               list(model.sparse_encoder.parameters()) +
               list(model.fusion1.parameters()) +
               list(model.fusion2.parameters()) +
               list(model.fusion3.parameters()) +
               list(model.head.parameters()),
     'lr': LR},
    {'params': model.refine.parameters(), 'lr': LR * 0.1},
], weight_decay=WEIGHT_DECAY)

    # CosineAnnealingWarmRestarts -- see model file for scheduler rationale
    scheduler = build_recommended_scheduler(
        optimizer,
        warm_restart_epochs=COSINE_T0,
        min_lr_floor=COSINE_ETA_MIN,
    )

    # ---- Checkpoint resume --------------------------------------------------
    RESUME_CHECKPOINT = None
    # RESUME_CHECKPOINT = CHECKPOINT_DIR / 'symgate_vdsr_epoch_050.pt'

    start_epoch    = 1
    best_metric    = float('inf')
    best_composite = float('inf')
    best_epoch     = 0
    stale_epochs   = 0
    train_history  = []

    if RESUME_CHECKPOINT is not None:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch    = ckpt['epoch'] + 1
        best_metric    = ckpt.get('best_metric', float('inf'))
        best_composite = ckpt.get('best_metric', float('inf'))
        best_epoch     = ckpt.get('best_epoch', 0)
        train_history  = ckpt.get('train_history', [])
        print(f'Resumed from epoch {ckpt["epoch"]} | best_composite={best_composite:.4f} @ ep {best_epoch}')
    else:
        print('Starting from scratch.')

    print(f'Train files: {len(train_loader.dataset)}')
    print(f'Val files:   {len(val_loader.dataset)}')

    epoch_bar = tqdm(range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True)

    val_stats = {
        'val_total': 0.0, 'val_mae': 0.0, 'val_rmse': 0.0, 'val_psnr': 0.0,
        'val_slope_rmse': 0.0, 'val_curve_rmse': 0.0, 'val_anchor_mae': 0.0, 'val_aux_mae': 0.0
    }
    composite = float('inf')
    
    for epoch in epoch_bar:
        t0 = time.time()

        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            train_loader=train_loader,
            device=device,
            grad_clip_norm=GRAD_CLIP_NORM,
            epoch=epoch,
            warmup_steps=WARMUP_STEPS,
            base_lr=LR,
            log_grad_norms=True,
        )
        
        # CONDITION: Validate every epoch up to 20, then every 3rd epoch.
        # If you want strictly NO validation before epoch 20, change this to:
        # should_validate = (epoch >= 20) and (epoch % 3 == 0)
        should_validate = (epoch <= 20) or ((epoch-20) % 3 == 0)

        is_best = False  # Default to False for epochs where we skip validation

        if should_validate:
            val_stats = validate_one_epoch(
                model=model,
                criterion=criterion,
                val_loader=val_loader,
                device=device,
                val_patch_batch=VAL_PATCH_BATCH,
            )
            
            # Composite: slope quality + Stream B own anchor quality.
            composite = val_stats['val_slope_rmse'] + val_stats['val_aux_mae']

            is_best = composite < best_composite
            if is_best:
                best_composite = composite
                best_metric    = composite
                best_epoch     = epoch
                stale_epochs   = 0
            else:
                # Only increment stale_epochs if we actually evaluated!
                stale_epochs += 1

        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        row = {
            'epoch':          epoch,
            'lr':             lr_now,
            'train_total':    train_stats['total'],
            'train_base':     train_stats['base'],
            'train_slope':    train_stats['slope'],
            'train_curve':    train_stats['curve'],
            'train_pin':      train_stats['pin'],
            'train_aux':      train_stats['aux'],
            'train_mae':      train_stats['mae'],
            'train_rmse':     train_stats['rmse'],
            'gn_dense':       train_stats['gn_dense'],
            'gn_sparse':      train_stats['gn_sparse'],
            'gn_fusion':      train_stats['gn_fusion'],
            'gn_refine':      train_stats['gn_refine'],
            'gn_head':        train_stats['gn_head'],
            # These will output the most recently calculated validation metrics 
            # if validation is skipped this epoch
            'val_total':      val_stats['val_total'],
            'val_mae':        val_stats['val_mae'],
            'val_rmse':       val_stats['val_rmse'],
            'val_psnr':       val_stats['val_psnr'],
            'val_slope_rmse': val_stats['val_slope_rmse'],
            'val_curve_rmse': val_stats['val_curve_rmse'],
            'val_anchor_mae': val_stats['val_anchor_mae'],
            'val_aux_mae':    val_stats['val_aux_mae'],
            'val_composite':  composite,
            'time_sec':       elapsed,
        }
        train_history.append(row)

        save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            is_best=is_best,
        )

        epoch_bar.set_postfix({
            'train': f"{train_stats['total']:.4f}",
            'slope': f"{val_stats['val_slope_rmse']:.4f}",
            'aux':   f"{val_stats['val_aux_mae']:.4f}",
            'best':  f"{best_composite:.4f}",
            'lr':    f"{lr_now:.2e}",
        })

        tqdm.write(
            f"Epoch {epoch:03d} | "
            f"TLoss {train_stats['total']:.4f} "
            f"(B={train_stats['base']:.3f} S={train_stats['slope']:.3f} "
            f"C={train_stats['curve']:.3f} P={train_stats['pin']:.3f} A={train_stats['aux']:.3f}) | "
            f"GN dense={train_stats['gn_dense']:.3f} sparse={train_stats['gn_sparse']:.3f} "
            f"fusion={train_stats['gn_fusion']:.3f} refine={train_stats['gn_refine']:.3f} | "
            f"ValMAE {val_stats['val_mae']:.4f} | "
            f"SlopeRMSE {val_stats['val_slope_rmse']:.4f} | "
            f"AuxMAE {val_stats['val_aux_mae']:.4f} | "
            f"AnchorMAE(sanity) {val_stats['val_anchor_mae']:.6f} | "
            f"Composite {composite:.4f} | "
            f"LR {lr_now:.2e} | "
            f"Best {best_composite:.4f} @ ep {best_epoch} | "
            f"time {elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}.")
            break

    epoch_bar.close()
    print('Training finished.')


In [ ]:
main()

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
Starting from scratch.
Train files: 1649
Val files:   6


Epochs:   0%|                                                                                  | 0/500 [00:00<…

Train:   0%|                                                                                   | 0/103 [00:46<…

Validate files:   0%|                                                                            | 0/6 [00:00<…

Patches 1/6:   0%|                                                                              | 0/61 [00:00<…

...

Train:   0%|                                                                                   | 0/103 [00:00<…

Validate files:   0%|                                                                            | 0/6 [00:00<…

Patches 1/6:   0%|                                                                              | 0/61 [00:00<…

Patches 2/6:   0%|                                                                              | 0/61 [00:00<…

Patches 3/6:   0%|                                                                              | 0/61 [00:00<…